# Tutorial 2: Gene (DNA) Embeddings

This notebook shows how to compute **DNA-level gene embeddings** with `embpy`.

Available DNA embedding models:

| Model key | Architecture | Notes |
|---|---|---|
| `enformer_human_rough` | Enformer | 896 output bins × 5 313 tracks → pooled |
| `borzoi_v0` / `borzoi_v1` | Borzoi | Similar genomic track prediction |
| `evo2_7b` / `evo2_40b` | Evo2 | DNA language model (requires GPU) |
| `evo2_7b_base` / `evo2_1b_base` | Evo2 (base) | Smaller checkpoints |

All models accept raw DNA sequences and return dense embeddings.
The `BioEmbedder` class handles sequence resolution and model dispatch
transparently.

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

In [ ]:
from embpy.embedder import BioEmbedder

# BioEmbedder auto-detects the best device (GPU if available)
embedder = BioEmbedder(device="auto")
print(f"Device: {embedder.device}")
print(f"Available models: {embedder.list_available_models()}")

## 1. Embed a single gene by symbol

Provide a human gene symbol and let `embpy` resolve the DNA sequence via
the Ensembl REST API, then embed it.

In [ ]:
import numpy as np

# Choose a model (use enformer_human_rough as the default example)
MODEL = "enformer_human_rough"

emb = embedder.embed_gene(
    identifier="TP53",
    model=MODEL,
    id_type="symbol",
    organism="human",
    pooling_strategy="mean",
)

print(f"Embedding shape: {emb.shape}")
print(f"Embedding dtype: {emb.dtype}")
print(f"First 10 values: {emb[:10]}")

## 2. Embed a gene by Ensembl ID

In [ ]:
emb_ens = embedder.embed_gene(
    identifier="ENSG00000141510",   # TP53
    model=MODEL,
    id_type="ensembl_id",
    pooling_strategy="mean",
)
print(f"Embedding shape: {emb_ens.shape}")

## 3. Embed a raw DNA sequence directly

If you already have a DNA sequence, you can pass it as the identifier
and the embedder will detect it automatically.

In [ ]:
# A synthetic DNA sequence (must be long enough for Enformer: 196_608 bp)
# For demonstration, we generate a random sequence.
rng = np.random.default_rng(42)
dna = "".join(rng.choice(list("ACGT"), size=196_608))

emb_raw = embedder.embed_gene(
    identifier=dna,
    model=MODEL,
    pooling_strategy="mean",
)
print(f"Raw DNA embedding shape: {emb_raw.shape}")

## 4. Batch embedding

Use `embed_genes_batch` to embed multiple genes efficiently. Failed
lookups return `None` without crashing the batch.

In [ ]:
genes = ["TP53", "BRCA1", "EGFR", "MYC"]

embeddings = embedder.embed_genes_batch(
    model=MODEL,
    identifiers=genes,
    id_type="symbol",
    organism="human",
    pooling_strategy="mean",
)

for gene, emb in zip(genes, embeddings):
    shape = emb.shape if emb is not None else "FAILED"
    print(f"  {gene:10s} → {shape}")

## 5. Pooling strategies

DNA models often produce multi-dimensional outputs (e.g. Enformer outputs
896 spatial bins × 5 313 tracks). The `pooling_strategy` parameter controls
how these are reduced to a single vector:

- `"mean"` – average over the spatial / sequence dimension
- `"max"` – max-pool over the spatial / sequence dimension
- `"cls"` – take the first token (for transformer models)

In [ ]:
for strategy in ["mean", "max"]:
    emb = embedder.embed_gene(
        "TP53", model=MODEL, pooling_strategy=strategy,
    )
    print(f"  pooling={strategy:5s} → shape {emb.shape}, mean={emb.mean():.4f}")

## 6. Using Evo2 (DNA language model)

Evo2 is a large DNA language model that requires a GPU. If a CUDA device
is available, you can use it like any other model.

In [ ]:
import torch

if torch.cuda.is_available():
    emb_evo2 = embedder.embed_gene(
        identifier="TP53",
        model="evo2_7b",
        id_type="symbol",
        pooling_strategy="mean",
    )
    print(f"Evo2 embedding shape: {emb_evo2.shape}")
else:
    print("No CUDA device found – skipping Evo2 (requires GPU).")

## 7. Comparing embeddings across models

Embeddings from different models live in different-dimensional spaces.
Later, in [Tutorial 6](06_combined_analysis.ipynb), we will show how to
align them via PCA.

In [ ]:
# Quick summary of dimension per model
for m in ["enformer_human_rough", "borzoi_v0"]:
    try:
        e = embedder.embed_gene("TP53", model=m, pooling_strategy="mean")
        print(f"  {m:30s} → dim {e.shape[0]:,}")
    except Exception as exc:
        print(f"  {m:30s} → {exc}")

## Summary

| Method | Use case |
|---|---|
| `embed_gene(identifier, model)` | Single gene (symbol, Ensembl ID, or raw DNA) |
| `embed_genes_batch(model, identifiers)` | Batch of genes |
| `pooling_strategy` kwarg | Control how spatial dims are reduced |

Next: [Tutorial 3 – Protein Embeddings](03_protein_embeddings.ipynb)